# Phase 1 — Environment Setup Check

Run every cell top-to-bottom. All cells should pass with no errors.
The final cell prints a summary table of every library and its version.

**Expected kernel:** `manifold-discovery`  
**Expected Python:** 3.10.x

> **First time?** Run this in your terminal before opening this notebook:
> ```bash
> conda env create -f environment.yml
> conda activate manifold-discovery
> python -m ipykernel install --user --name manifold-discovery --display-name "Manifold Discovery"
> python src/utils/setup_project.py
> ```

In [ ]:
# Cell 1 — Python version gate
import sys
assert sys.version_info >= (3, 10), f"Need Python ≥ 3.10, got {sys.version}"
print(f"✅  Python {sys.version.split()[0]}")

In [ ]:
# Cell 2 — Core scientific stack
import numpy as np
import scipy
import pandas as pd
import sklearn

for name, mod in [("numpy", np), ("scipy", scipy), ("pandas", pd), ("scikit-learn", sklearn)]:
    print(f"✅  {name:<16} {mod.__version__}")

In [ ]:
# Cell 3 — Visualisation libraries
import matplotlib
import seaborn as sns
import plotly

print(f"✅  matplotlib     {matplotlib.__version__}")
print(f"✅  seaborn        {sns.__version__}")
print(f"✅  plotly         {plotly.__version__}")

In [ ]:
# Cell 4 — Manifold learning methods
from sklearn.manifold import Isomap, LocallyLinearEmbedding, SpectralEmbedding, TSNE
from sklearn.decomposition import PCA
import umap

print("✅  sklearn manifold: Isomap, LLE, SpectralEmbedding, TSNE")
print(f"✅  umap-learn      {umap.__version__}")

In [ ]:
# Cell 5 — PyTorch + device detection
import torch

cuda_ok = torch.cuda.is_available()
mps_ok  = hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()
DEVICE  = 'cuda' if cuda_ok else 'mps' if mps_ok else 'cpu'

print(f"✅  torch          {torch.__version__}")
print(f"   CUDA available : {cuda_ok}")
print(f"   MPS  available : {mps_ok}")
print(f"   → Autoencoder will train on: '{DEVICE}'")

In [ ]:
# Cell 6 — Project folder structure check
from pathlib import Path

ROOT = Path('..').resolve()
required = [
    'data', 'notebooks',
    'src/datasets', 'src/methods', 'src/evaluation', 'src/utils',
    'results/figures', 'results/metrics', 'results/models',
]

all_ok = True
for rel in required:
    d = ROOT / rel
    ok = d.exists()
    print(f"{'✅' if ok else '❌'}  {rel}")
    if not ok:
        all_ok = False

if not all_ok:
    print("\n⚠️  Run: python ../src/utils/setup_project.py  to create missing dirs.")
else:
    print("\n✅  All project directories present.")

In [ ]:
# Cell 7 — Smoke test: generate + render Swiss Roll
import matplotlib.pyplot as plt
from sklearn.datasets import make_swiss_roll

X, t = make_swiss_roll(n_samples=1500, noise=0.1, random_state=42)

fig = plt.figure(figsize=(7, 5))
ax  = fig.add_subplot(111, projection='3d')
sc  = ax.scatter(X[:, 0], X[:, 1], X[:, 2], c=t, cmap='viridis', s=8, alpha=0.7)
plt.colorbar(sc, ax=ax, label='Manifold parameter t')
ax.set_title('Swiss Roll — smoke test', fontsize=13)
ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
plt.tight_layout()

out_path = ROOT / 'results' / 'figures' / '00_smoke_test_swiss_roll.png'
plt.savefig(out_path, dpi=150)
plt.show()

print(f"✅  Swiss Roll: shape={X.shape}, t∈[{t.min():.2f}, {t.max():.2f}]")
print(f"   Saved to: {out_path.relative_to(ROOT)}")

In [ ]:
# Cell 8 — Full version summary table
import importlib

checks = [
    ('numpy',        'numpy'),
    ('scipy',        'scipy'),
    ('pandas',       'pandas'),
    ('scikit-learn', 'sklearn'),
    ('matplotlib',   'matplotlib'),
    ('seaborn',      'seaborn'),
    ('plotly',       'plotly'),
    ('umap-learn',   'umap'),
    ('torch',        'torch'),
    ('streamlit',    'streamlit'),
    ('tqdm',         'tqdm'),
]

print(f"{'Library':<18} {'Version':<14} Status")
print('─' * 46)
all_installed = True
for name, mod in checks:
    try:
        m   = importlib.import_module(mod)
        ver = getattr(m, '__version__', 'n/a')
        print(f"{name:<18} {ver:<14} ✅")
    except ImportError:
        print(f"{name:<18} {'missing':<14} ❌")
        all_installed = False

print(f"\nTraining device : {DEVICE}")

if all_installed:
    print("\n🎉  Environment fully verified — proceed to 01_datasets.ipynb")
else:
    print("\n⚠️  Fix missing packages, then re-run this notebook.")